In [ ]:
import re
import os
import logging
import time
from datetime import datetime

logger = logging.getLogger(__name__)
if not logger.handlers:
    handler = logging.FileHandler('log_lines.txt')
    formatter = logging.Formatter('%(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

def get_first_n_words(s, n):
    """Function to get the first n words from a given string s"""
    words = s.split()[:n]
    return ' '.join(words) if len(words) == n else '[short sentence]'

def get_logger(file_name):
    """Function to setup logger for each input file"""
    main_dir = '.' # replace with your main directory path if needed
    base_name = os.path.basename(file_name)
    name, _ = os.path.splitext(base_name)
    new_dir = os.path.join(main_dir, 'noBlanks', name + '_log')
    os.makedirs(new_dir, exist_ok=True)
    log_file_name = os.path.join(new_dir, name + '_log.txt')
    i = 1
    while os.path.isfile(log_file_name):
        log_file_name = os.path.join(new_dir, f'{name}_log-{i}.txt')
        i += 1
    logger = logging.getLogger(file_name)
    if not logger.handlers:
        handler = logging.FileHandler(log_file_name)
        formatter = logging.Formatter('%(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

def log_separator(logger):
    """Function to write separator line to the log file"""
    logger.info('-' * 80)

def log_header(logger, file_name):
    """Function to write header lines to the log file"""
    logger.info('=' * 80)
    logger.info(f'Time:\t{datetime.now().strftime("%H:%M:%S")}\t\tDate:\t{datetime.now().strftime("%Y-%m-%d")}')
    logger.info(f'Call:\tline_condenser()')
    logger.info(f'File:\t{file_name}')
    log_separator(logger)

def log_footer(logger, total_changes, process_time):
    """Function to write footer lines to the log file"""
    log_separator(logger)
    logger.info(f'Total changes made:\t{total_changes}')
    logger.info(f'Finished process in:\t{process_time * 1000:.3f} ms')
    log_separator(logger)

def get_output_file_name(file_name):
    """Function to generate the output file name"""
    main_dir = '.' # replace with your main directory path if needed
    base_name = os.path.basename(file_name)
    name, ext = os.path.splitext(base_name)
    new_dir = os.path.join(main_dir, 'noBlanks')
    os.makedirs(new_dir, exist_ok=True)
    output_file_name = os.path.join(new_dir, name + '_noBlanks' + ext)
    i = 1
    while os.path.isfile(output_file_name):
        output_file_name = os.path.join(new_dir, f'{name}_noBlanks-{i}{ext}')
        i += 1
    return output_file_name

def line_condenser(file_name):
    """Main function to condense lines in the input file"""
    start_time = time.time()
    total_changes = 0

    individual_logger = get_logger(file_name)
    log_header(logger, file_name)
    log_header(individual_logger, file_name)

    with open(file_name, 'r') as f:
        lines = f.readlines()

    output_lines = []
    last_speaker = None

    for i, line in enumerate(lines):
        line = line.rstrip() # remove trailing whitespaces/newlines

        # If the line is empty, skip it
        if line == '':
            total_changes += 1
            continue

        # If the line starts with a speaker's name
        if re.match(r'^[A-Z]+:', line):
            output_lines.append(line)
            last_speaker = line.split(':')[0]
        # If the line is a continuation of the last speaker's text
        elif last_speaker:
            output_lines[-1] += ' ' + line
            total_changes += 1
            individual_logger.info(f"Line {i+1} starting with: '{get_first_n_words(line, 3)}' moved to Line {len(output_lines)} starting with: '{get_first_n_words(output_lines[-1], 3)}'")

    # Writing the output file
    output_file_name = get_output_file_name(file_name)
    with open(output_file_name, 'w') as f:
        f.write('\n'.join(output_lines))

    log_footer(logger, total_changes, time.time() - start_time)
    log_footer(individual_logger, total_changes, time.time() - start_time)

    print(f'Output written to {output_file_name}')

file = "all_debates/2016/preclean/preclean-2016-09-26-debate.txt" 
line_condenser(file)


In [ ]:
import os

def do(path):
    """Run line_condenser on all text files in a directory (and its subdirectories) 
    that start with "preclean-" in their name."""

    if os.path.isfile(path):
        if path.endswith('.txt') and os.path.basename(path).startswith('preclean-'):
            line_condenser(path)
    else:
        for dirpath, dirnames, filenames in os.walk(path):
            for file in filenames:
                if file.endswith('.txt') and file.startswith('preclean-'):
                    file_path = os.path.join(dirpath, file)
                    line_condenser(file_path)


In [ ]:
import os

debates_dir = "all_debates"

for root, dirs, files in os.walk(debates_dir):
    if 'preclean' in dirs:
        preclean_dir = os.path.join(root, 'preclean')
        for file_name in os.listdir(preclean_dir):
            file_path = os.path.join(preclean_dir, file_name)
            if os.path.isfile(file_path) and file_name.endswith('.txt'):
                line_condenser(file_path)


In [4]:
import os
import re
import logging
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.probability import FreqDist

def debate_stats(directory):
    # logging
    logging.basicConfig(filename='debate_stat_error_log.txt', level=logging.ERROR)

    total_words = 0
    total_words_no_speaker_no_insertions = 0
    total_interruptions = 0
    total_speakers = set()
    speaker_stats = defaultdict(lambda: {'turns': 0, 'words': 0, 'sentences': 0, 'interrupts': 0})

    stop_words = set(stopwords.words('english'))

    all_files = []

    file_count = 0

    for subdir, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".txt"):
                file_count += 1

                file_name = os.path.join(subdir, file)

                with open(file_name, 'r') as f:
                    lines = f.read().split('\n')

                all_files.extend(lines)

                unique_speakers = set()

                file_total_words = 0
                file_total_words_no_speaker_no_insertions = 0
                file_total_interruptions = 0

                for line in lines:
                    try:
                        speaker_name, text = line.split(':', 1)
                    except ValueError as e:
                        logging.error(f'Error at file {file_name} line {line}: {str(e)}')
                        print(f'|- {file_name} -|  |- {line} -|')
                        continue

                    unique_speakers.add(speaker_name)
                    total_speakers.add(speaker_name)

                    words = text.split()
                    file_total_words += len(words)
                    file_total_words_no_speaker_no_insertions += len([word for word in words if not re.match(r"\[.*\]|\(.*\)", word)])

                    if '-' in text or 'interruption' in text or 'crosstalk' in text:
                        file_total_interruptions += 1

                    sentences = text.split('. ')
                    speaker_stats[speaker_name]['turns'] += 1
                    speaker_stats[speaker_name]['words'] += len(words)
                    speaker_stats[speaker_name]['sentences'] += len(sentences)
                    if '-' in text or 'interruption' in text or 'crosstalk' in text:
                        speaker_stats[speaker_name]['interrupts'] += 1

                total_words += file_total_words
                total_words_no_speaker_no_insertions += file_total_words_no_speaker_no_insertions
                total_interruptions += file_total_interruptions

    results = []
    results.append(f"Number of debates: {file_count}")
    results.append(f"Total unique speakers: {len(total_speakers)}")
    results.append(f"Total words: {total_words}")
    results.append(f"Total words (no speaker names or insertions): {total_words_no_speaker_no_insertions}")
    results.append(f"Total interruptions: {total_interruptions}")
    results.append(f"Most common words in all debates: {get_most_common_words(' '.join(all_files), 20)}")

    for speaker, stats in speaker_stats.items():
        results.append(f"Speaker: {speaker}")
        results.append(f"Turns: {stats['turns']}")
        results.append(f"Words: {stats['words']}")
        results.append(f"Average words per turn: {stats['words'] / stats['turns']}")
        results.append(f"Sentences: {stats['sentences']}")
        results.append(f"Average sentences per turn: {stats['sentences'] / stats['turns']}")
        results.append(f"Interruptions: {stats['interrupts']}")

    return results

def get_most_common_words(text, num=20):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if not word in stop_words]
    fdist = FreqDist(words)
    return fdist.most_common(num)

def writeitdown(func):
    import datetime
    output_folder = func.__name__
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)
    filename = func.__name__ + "-" + datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S") + ".txt"
    filepath = os.path.join(output_folder, filename)
    with open(filepath, 'w') as f:
        f.write('\n'.join(func()))


ERROR: Could not find a version that satisfies the requirement prettyable (from versions: none)
ERROR: No matching distribution found for prettyable


ModuleNotFoundError: No module named 'prettytable'

In [ ]:
debate_stats("noBlanks")